# AI Incident Replay Studio
### Portfolio Presentation · Visual Incident Investigation Platform

---

> *A cinematic, time-based replay engine for exploring how distributed system incidents unfold — built as a full-stack, Dockerized demo platform.*

---

## 1. Project Goal

**AI Incident Replay Studio** answers a deceptively simple question:

> *"What exactly happened during this system incident, and in what order?"*

In real-world operations, incidents are messy. Logs pile up. Services fail in cascades. Root cause is hard to trace. Post-mortems are written after the fact, often missing the sequence of events.

This project builds a **visual investigation interface** where you can:
- Load a structured incident scenario
- Replay it second by second on a live service dependency graph
- Watch services transition from healthy → degraded → critical → recovering
- Follow an AI-style narrative that explains what happened
- Hear sound cues when critical events fire

The goal is **clarity through replay** — turning raw incident data into a cinematic, investigable experience.

## 2. Why This Project Is Unique

Most incident tooling focuses on alerting or dashboards. This project focuses on **post-incident investigation and learning**.

| What makes it stand out | Details |
|---|---|
| **Time-based replay** | Scrub forward and back through incident events like a video player |
| **Live graph state** | Service nodes change color and glow in real time during replay |
| **AI narrative layer** | Rule-based analysis delivers root cause, impact, and operator notes |
| **Sound design** | Synthesized audio cues respond to event severity (warning tones, critical crashes) |
| **Dockerized full-stack** | Entire platform runs with a single `docker-compose up` |
| **Multiple incidents** | Switch between different incident scenarios and replay them independently |

It combines **observability tooling UX**, **graph visualization**, and **sound design** in a single cohesive demo — a combination that is uncommon in portfolio projects.

## 3. Architecture Overview

```
┌─────────────────────────────────────────────────────────────┐
│                     Docker Compose                          │
│                                                             │
│   ┌──────────────────┐         ┌──────────────────────┐    │
│   │   Frontend       │  HTTP   │   Backend API        │    │
│   │   React + Vite   │ ──────► │   Python FastAPI     │    │
│   │   Port :5173     │         │   Port :8000         │    │
│   └──────────────────┘         └──────────────────────┘    │
│                                         │                   │
│                                         ▼                   │
│                               ┌─────────────────┐          │
│                               │  incidents.json  │         │
│                               │  (scenario data) │         │
│                               └─────────────────┘          │
└─────────────────────────────────────────────────────────────┘
```

**Stack summary:**
- **Frontend**: React 18 + TypeScript + Vite + ReactFlow
- **Backend**: Python 3.11 + FastAPI + Uvicorn
- **Data**: JSON-based incident scenarios with structured event timelines
- **Runtime**: Docker + Docker Compose (no external services required)
- **Audio**: Web Audio API (synthesized, no audio files needed)
- **Graph**: ReactFlow for animated service dependency visualization

## 4. Core Features

### Replay Engine
The replay engine advances through incident events at configurable speed (1×, 2×, 4×). Each tick evaluates which events have occurred and updates the graph state accordingly. You can also scrub manually via the timeline slider.

### Service Dependency Graph
Services are rendered as nodes in a connected graph (frontend → api → db, cache, queue → worker). Each node's color reflects the service state at the current replay timestamp:

| State | Color | Meaning |
|---|---|---|
| `healthy` | Steel blue | No events yet |
| `warning` | Amber | Latency spike, degradation, surge |
| `critical` | Red + pulse | Service crash, timeout, queue overflow |
| `recovering` | Green | Recovery event detected |

### AI Narrative Panel
Each incident includes a structured analysis delivered by the backend:
- **Summary**: What happened at a high level
- **Root Cause**: The originating failure
- **Impact**: What was affected
- **Operator Note**: What action was taken to resolve

### Event Feed
A live-updating feed of events as they occur during replay. Each card is color-coded by severity, with a slide-in animation for new events.

### Summary Panel
When replay completes, a post-replay summary slides in with key incident statistics and a final narrative.

## 5. Replay Workflow

The full investigation flow from start to summary:

```
1. Select incident from dropdown
        │
        ▼
2. Frontend fetches incident data + AI analysis from backend
        │
        ▼
3. Timeline starts at T+0s, all services show as healthy
        │
        ▼
4. Press ▶ Play — time advances, events fire in order
        │
        ├── Service nodes change state and color
        ├── Event cards slide into the feed
        ├── Alert banner updates with latest event
        ├── Sound cue fires (warning / critical tone)
        └── Critical popup appears on service_crash events
        │
        ▼
5. Replay reaches max time → auto-stop
        │
        ▼
6. Summary panel animates in — post-mortem view
        │
        ▼
7. Reset and replay with different speed, or switch incident
```

## 6. Backend API Overview

The backend is a lightweight FastAPI service that serves incident data and pre-computed analysis.

```python
# Core endpoints
GET  /incidents                          # List all incidents
GET  /incidents/{incident_id}            # Single incident with events
GET  /incidents/{incident_id}/analysis   # AI narrative for the incident
```

**Example incident structure:**

```json
{
  "incident_id": "INC-001",
  "title": "API Database Connection Failure",
  "severity": "critical",
  "status": "resolved",
  "services": ["frontend", "api", "db", "cache"],
  "events": [
    { "time": 0,  "service": "api",      "event": "latency_spike",   "message": "API latency elevated to 2400ms" },
    { "time": 15, "service": "db",       "event": "service_crash",   "message": "Database connection pool exhausted" },
    { "time": 45, "service": "api",      "event": "timeout",         "message": "API requests timing out" },
    { "time": 90, "service": "db",       "event": "recovery",        "message": "Database connections restored" }
  ]
}
```

The backend is intentionally simple — its value is not in complex logic but in providing a clean, fast API that the frontend can query to build the full replay experience.

## 7. Frontend Investigation Experience

The frontend is a **single-component React application** (by design — no over-engineering for a demo). It manages all replay state internally with React hooks.

### Key state variables:

```typescript
currentTime   // Current replay timestamp in seconds
isPlaying     // Whether time is advancing
speed         // Replay speed multiplier (1x / 2x / 4x)
visibleEvents // Events that have occurred up to currentTime
showSummary   // Whether the post-replay panel is visible
```

### Event-driven state transitions:
Each event in the timeline maps to a visual state change. The `getServiceState()` function scans all events up to `currentTime` and returns the current state for any service:

```typescript
function getServiceState(events, service, currentTime) {
  // Find the latest event for this service at or before currentTime
  // Return 'critical' | 'warning' | 'recovering' | 'healthy'
}
```

This drives both the **node color** and the **glow effect** in the ReactFlow graph — creating the live, responsive feel of the investigation experience.

## 8. AI Narrative Concept

The "AI" in AI Incident Replay Studio refers to the **narrative layer** — pre-structured, human-readable analysis attached to each incident.

This is **intentionally honest**: the analysis is rule-based and pre-authored, not generated by a live language model. This makes the system:
- **Deterministic** — same incident always produces the same analysis
- **Reliable** — no API rate limits, no hallucinations, no latency
- **Demonstrable** — the analysis quality is controlled and purposeful

The concept is designed to show *what AI-augmented incident analysis could look like* in a real observability tool — where a language model would generate these narratives dynamically from raw telemetry.

This framing positions the project as a **prototype and design exploration**, not a production system claim. That is its strength — it demonstrates the UX pattern clearly without overpromising.

## 9. Sound & Cinematic Layer

One of the most distinctive aspects of the project is its **synthesized sound design**.

Sound cues are generated entirely in-browser using the **Web Audio API** — no audio files, no external dependencies.

| Sound | Trigger | Waveform |
|---|---|---|
| `start` | Replay begins | Sine wave, rising pitch |
| `warning` | Latency / degradation event | Triangle wave, oscillating |
| `critical` | Service crash | Sawtooth wave, descending |
| `complete` | Replay finishes | Sine wave, ascending chord |

The sound system is opt-in (muted by default). When enabled, it turns the visual replay into a **multi-sensory investigation experience** — which is the cinematic quality the project is designed to evoke.

This is a deliberate design choice: real incident response is high-stakes, often chaotic. The audio layer reinforces that emotional register without being gratuitous.

## 10. Docker Runtime

The full platform runs in Docker with a single command:

```bash
docker-compose up
```

**docker-compose.yml structure:**

```yaml
services:
  backend:
    build: ./backend
    ports:
      - "8000:8000"

  frontend:
    build: ./frontend
    ports:
      - "5173:5173"
    environment:
      - VITE_API_BASE_URL=http://localhost:8000
```

Both services build and start independently. The frontend proxies API calls to the backend using an environment variable, making the setup portable across machines.

**Local development** is also supported with `npm run dev` (frontend) and `uvicorn main:app --reload` (backend) for fast iteration cycles.

## 11. Portfolio Value

This project demonstrates a range of skills that are relevant to modern software roles:

### Full-Stack Integration
React frontend communicating with a Python API backend, both containerized and coordinated via Docker Compose.

### State Management & Event-Driven UI
Complex time-based state logic: replay playback, speed control, event visibility, alert level transitions — all managed cleanly with React hooks.

### Graph Visualization
Service dependency graphs rendered with ReactFlow, with dynamic node states, animated edges, and real-time visual feedback.

### API Design
Clean REST API with typed response models, CORS handling, and structured data schemas — production-pattern architecture.

### UX Thinking
The UI is designed around an investigation workflow — not just "show data". Alert banners, sound cues, slide-in animations, and a post-replay summary all serve a deliberate user journey.

### Sound Synthesis
Web Audio API usage is uncommon in portfolio projects. It shows curiosity, creativity, and the ability to work with browser APIs beyond DOM manipulation.

## 12. Why This Project Stands Out

---

Most portfolio projects fall into predictable patterns: a CRUD app, a weather widget, a to-do list. AI Incident Replay Studio is different because it builds something **purposeful and uncommon**.

**It solves a real problem class.** Incident investigation is a genuine pain point in software engineering. Post-mortems are often written from memory. This project shows what a better tool could look like.

**It combines disciplines.** Frontend, backend, DevOps (Docker), audio engineering, data visualization, UX design — all in one coherent project. That breadth is deliberate.

**It has a clear narrative.** You can demo this in two minutes and a viewer immediately understands what it does and why it matters. Most portfolio projects require extensive explanation.

**It is honest about its scope.** It presents itself as a well-built demo platform, not an enterprise product. That intellectual honesty is itself a professional quality.

**It is visually polished.** The dark slate UI, animated graph, cinematic event feed, and synthesized audio cues give it production-level visual quality — rare in personal projects.

---

> *Built as a demonstration of full-stack product thinking — from data architecture to visual investigation experience.*

---

## 13. Final Summary

| | |
|---|---|
| **Project** | AI Incident Replay Studio |
| **Type** | Full-stack demo platform |
| **Frontend** | React 18 + TypeScript + ReactFlow |
| **Backend** | Python FastAPI |
| **Deployment** | Docker + Docker Compose |
| **Key feature** | Time-based incident replay with live service graph |
| **Audio** | Synthesized via Web Audio API |
| **AI layer** | Rule-based narrative analysis |
| **Status** | Complete demo platform — fully functional |

### What was built:
1. A structured incident data schema with multiple scenario timelines
2. A FastAPI backend serving incident data and analysis
3. A React frontend with time-based replay engine
4. A live ReactFlow service dependency graph with dynamic state
5. An event feed with severity-coded, animated event cards
6. An AI narrative panel with summary, root cause, and operator notes
7. A synthesized sound system for critical event cues
8. A post-replay summary panel with incident statistics
9. Full Docker containerization for portable deployment
10. A premium dark UI designed for cinematic investigation UX

---

*AI Incident Replay Studio — Portfolio project demonstrating full-stack product engineering.*